# 01 · Data preparation and audit
Philippine Inflation Observatory. Run this notebook before model development.
The supplied files are treated as source data, not as instructions.
Raw files remain unchanged. See `docs/data-model.md` for the ERD and fact grains.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
if (ROOT/'.packages').exists():
    sys.path.insert(0,str(ROOT/'.packages'))
import pandas as pd
import numpy as np

## Parse source formats
National all-items inflation is already a year-on-year percent. Read monthly average PHP/USD, month-end policy decisions, and DCS M3 levels (million PHP). Revised and provisional M3 date labels are normalized without discarding their original workbook.
The numeric policy parser carries previous rates over five nonnumeric 2011 hold decisions. Review their descriptions below.

In [2]:
from src.prepare import prepare
monthly, manifest = prepare()
display(monthly.head(), monthly.tail())
display(pd.DataFrame(manifest['coverage']).T)
display(pd.read_csv(ROOT/'data/processed/policy_review.csv'))

,inflation,exchange_rate,m3,policy_rate,m3_growth
date,,,,,
2010-01-01,3.9,46.027600,3.910182e+06,4.0,NaN
2010-02-01,3.9,46.311600,3.939374e+06,4.0,NaN
2010-03-01,3.9,45.742478,3.991470e+06,4.0,NaN
2010-04-01,4.0,44.626579,4.025107e+06,4.0,NaN
2010-05-01,3.9,45.597105,4.046092e+06,4.0,NaN


,inflation,exchange_rate,m3,policy_rate,m3_growth
date,,,,,
2026-04-01,7.2,60.291316,2.034729e+07,4.50,12.241121
2026-05-01,6.8,61.441000,2.060313e+07,4.50,12.798545
2026-06-01,6.4,61.251286,2.053946e+07,4.75,10.739871
2026-07-01,6.2,61.596261,2.051968e+07,4.75,10.295775
2026-08-01,6.1,61.328105,NaN,5.00,NaN


,start,end,missing
inflation,2010-01-01 00:00:00,2026-08-01 00:00:00,0
exchange_rate,2010-01-01 00:00:00,2026-08-01 00:00:00,0
m3,2010-01-01 00:00:00,2026-07-01 00:00:00,1
policy_rate,2010-01-01 00:00:00,2026-08-01 00:00:00,0
m3_growth,2011-01-01 00:00:00,2026-07-01 00:00:00,13


,row,date,description
0,119,2011-12-01,The Monetary Board decided to maintain the ove...
1,120,2011-10-20,The Monetary Board decided to maintain the ove...
2,121,2011-09-08,The Monetary Board decided to keep the overnig...
3,122,2011-07-28,The Monetary Board decided to maintain the BSP...
4,123,2011-06-16,The Monetary Board decided to keep policy rate...


## Validate grain, continuity and completeness
The wide model table has one national reference month per row. The database stores one indicator × geography × reference month × source vintage per observation. No target or M3 interpolation is allowed.

In [3]:
primary = monthly.loc['2010-01-01':'2026-07-01']
assert len(primary) == 199
assert primary.index.is_unique
assert primary.index.equals(pd.date_range('2010-01-01','2026-07-01',freq='MS',name='date'))
assert primary[['inflation','exchange_rate','m3','policy_rate']].notna().all().all()
assert (primary.m3 > 0).all()
assert np.isclose(monthly.loc['2026-07-01','m3_growth'],100*(monthly.loc['2026-07-01','m3']/monthly.loc['2025-07-01','m3']-1))
print('Passed: 199 continuous primary monthly rows; source series complete.')

Passed: 199 continuous primary monthly rows; source series complete.


## Explore indicators
These plots show association and history, not causal relationships. M3 growth is derived after month alignment. Its first 12 rows are missing because a full prior year is required.

In [4]:
import plotly.express as px
for col in ['inflation','exchange_rate','policy_rate','m3_growth']:
    display(px.line(monthly.reset_index(),x='date',y=col,title=col.replace('_',' ').title()))

## Provenance and research limitations
File SHA-256 values identify uploaded vintages. Original release dates and historical vintages are absent, so availability dates remain unknown. The second notebook uses a conservative delay assumption and reports retrospective evaluation. Unemployment was not supplied. `ms.xls` is retained for reconciliation, not merged as a duplicate M3 series.

In [5]:
display(pd.DataFrame(manifest['files']))
print(chr(10).join(manifest['notes']))

,file,sha256,bytes
0,Dataset-11995-2018.csv,b02d6827f33ef679121e2fa3b773157977bf9bbb2d3229...,4553
1,Dataset-12019-2026.csv,03d91d90472d2af5806261f44f79db91012a62e0f0e2a2...,1629
2,dcs_srf.xlsx,d4f502ffc57c248e82f9c97726ce91d62cf936e2409abc...,280249
3,MonetaryPolicyDecisions.xlsx,9837b1f40cc0b59b27193f641a89c962ee70a0c4c3e019...,76050
4,ms.xls,1f5f98566c016c0ab52197d0c690855bbbe0c888131697...,289280
5,pesodollar.xlsx,077dccd10c75b255b306f3d2973f3b1b43986b549bcff9...,205854


User-supplied source files; not independently authenticated.
Source vintage is the supplied file hash; original release dates are unavailable.
Policy rate extracted from decision text: review policy_review.csv before research use.
ms.xls retained for reconciliation; primary M3 comes from DCS level row 25.
Unemployment not supplied.
Model experiment ends July 2026; August inflation is available for display.
